## Notebook 3 - Supervised Model using CatBoost

This notebook focuses on predicting the `compatibility_score` between user pairs (`src_user_id`, `dst_user_id`) using a supervised machine learning approach. The task is treated as a **regression problem** because the `compatibility_score` is a continuous numerical value ranging from 0 to 1.

### Model Selection: CatBoostRegressor

**CatBoostRegressor** was chosen for its suitability for tabular data and its robust handling of categorical features. It automatically processes categorical variables, reducing the need for extensive manual preprocessing. Additionally, CatBoost is known for its strong performance and practical application in real-world scenarios.

### Features Used

Pairwise features are engineered to capture the similarity and differences between the source and destination profiles. These include:

*   **Numerical Differences**: `age_diff`, `company_size_diff`.
*   **Categorical Matches**: Binary indicators like `gender_match`, `role_match`, `seniority_match`, `industry_match`, `city_match`, and `company_match`.
*   **Text Overlap (Jaccard Similarity)**: `interest_overlap`, `objective_overlap`, `constraint_overlap` (based on semicolon-separated text fields).
*   **Raw Text Fields**: Cleaned versions of `src_interests`, `dst_interests`, `src_objectives`, `dst_objectives`, `src_constraints`, `dst_constraints` for CatBoost to learn from directly.
*   **Extra Categorical Text**: `src_role`, `dst_role`, `src_industry`, `dst_industry`.

### Validation and Early Stopping

The dataset is split into training and validation sets using `train_test_split` (with a 15% validation size and `random_state=42`) to evaluate the model's generalization performance. **Early stopping** is employed during model training (`early_stopping_rounds=200`). This technique monitors the model's performance on the validation set and stops training if the performance does not improve for a specified number of iterations, preventing overfitting and optimizing training time.

### Saved Artifacts

After training, the following files are saved for future use:

*   `catboost_model.cbm`: The trained CatBoostRegressor model.
*   `feature_columns.pkl`: A serialized list of the feature column names used during training, essential for ensuring consistent feature ordering during inference.

In [ ]:
!pip -q install catboost openpyxl joblib tqdm

import pandas as pd
import numpy as np
import re
import joblib
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor, Pool


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.5 MB/s eta 0:00:00


In [ ]:
train = pd.read_excel("train.xlsx")
test  = pd.read_excel("test.xlsx")
target = pd.read_csv("target.csv")

all_profiles = pd.concat([train, test], axis=0).reset_index(drop=True)

print(train.shape, test.shape, target.shape)


(600, 12) (400, 12) (360000, 3)


In [ ]:
for col in all_profiles.columns:
    if all_profiles[col].dtype == "object":
        all_profiles[col] = all_profiles[col].fillna("unknown")
    else:
        all_profiles[col] = all_profiles[col].fillna(all_profiles[col].median())


In [ ]:
def clean_text(x):
    x = str(x).lower()
    x = x.replace(";", " ")
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def split_semicolon(x):
    if pd.isna(x):
        return []
    parts = [p.strip().lower() for p in str(x).split(";")]
    return [p for p in parts if p]

def jaccard_semicolon(a, b):
    A = set(split_semicolon(a))
    B = set(split_semicolon(b))
    if len(A) == 0 and len(B) == 0:
        return 0.0
    return len(A.intersection(B)) / max(1, len(A.union(B)))


In [ ]:
profile_dict = all_profiles.set_index("Profile_ID").to_dict(orient="index")
print("Profiles stored:", len(profile_dict))



Profiles stored: 1000


In [ ]:
def make_pair_features(src_id, dst_id):
    src = profile_dict[src_id]
    dst = profile_dict[dst_id]

    row = {}

    # Numeric diffs
    row["age_diff"] = abs(src["Age"] - dst["Age"])
    row["company_size_diff"] = abs(src["Company_Size_Employees"] - dst["Company_Size_Employees"])

    # Categorical matches
    row["gender_match"] = int(src["Gender"] == dst["Gender"])
    row["role_match"] = int(src["Role"] == dst["Role"])
    row["seniority_match"] = int(src["Seniority_Level"] == dst["Seniority_Level"])
    row["industry_match"] = int(src["Industry"] == dst["Industry"])
    row["city_match"] = int(src["Location_City"] == dst["Location_City"])
    row["company_match"] = int(src["Company_Name"] == dst["Company_Name"])

    # Text overlap (important)
    row["interest_overlap"] = jaccard_semicolon(src["Business_Interests"], dst["Business_Interests"])
    row["objective_overlap"] = jaccard_semicolon(src["Business_Objectives"], dst["Business_Objectives"])
    row["constraint_overlap"] = jaccard_semicolon(src["Constraints"], dst["Constraints"])

    # Raw text fields (CatBoost can use them as categorical)
    row["src_interests"] = clean_text(src["Business_Interests"])
    row["dst_interests"] = clean_text(dst["Business_Interests"])

    row["src_objectives"] = clean_text(src["Business_Objectives"])
    row["dst_objectives"] = clean_text(dst["Business_Objectives"])

    row["src_constraints"] = clean_text(src["Constraints"])
    row["dst_constraints"] = clean_text(dst["Constraints"])

    # Extra: role/industry text
    row["src_role"] = str(src["Role"]).lower()
    row["dst_role"] = str(dst["Role"]).lower()

    row["src_industry"] = str(src["Industry"]).lower()
    row["dst_industry"] = str(dst["Industry"]).lower()

    return row



In [ ]:
      X_rows = []
y = []

for r in tqdm(target.itertuples(), total=len(target)):
    X_rows.append(make_pair_features(r.src_user_id, r.dst_user_id))
    y.append(r.compatibility_score)

X = pd.DataFrame(X_rows)
y = np.array(y)

print("Train pairs:", X.shape)


100%|██████████| 360000/360000 [00:20<00:00, 17732.10it/s]


Train pairs: (360000, 21)


In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42
)

cat_features = [i for i, c in enumerate(X.columns) if X[c].dtype == "object"]

train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool   = Pool(X_val, y_val, cat_features=cat_features)

model = CatBoostRegressor(
    loss_function="RMSE",
    iterations=1500,
    learning_rate=0.08,
    depth=7,
    l2_leaf_reg=6,
    random_seed=42,
    verbose=200
)

model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True,
    early_stopping_rounds=200
)

val_pred = np.clip(model.predict(X_val), 0, 1)

rmse = np.sqrt(mean_squared_error(y_val, val_pred))
mse  = mean_squared_error(y_val, val_pred)

print("Validation RMSE:", rmse)
print("Validation MSE :", mse)


0:	learn: 0.0750171	test: 0.0750325	best: 0.0750325 (0)	total: 1.03s	remaining: 25m 46s
200:	learn: 0.0120837	test: 0.0095807	best: 0.0095807 (200)	total: 2m 47s	remaining: 18m 1s
400:	learn: 0.0111098	test: 0.0084117	best: 0.0084117 (400)	total: 5m 24s	remaining: 14m 50s
600:	learn: 0.0106019	test: 0.0078660	best: 0.0078660 (600)	total: 7m 58s	remaining: 11m 55s
800:	learn: 0.0103164	test: 0.0075616	best: 0.0075616 (800)	total: 10m 34s	remaining: 9m 13s
1000:	learn: 0.0100301	test: 0.0072444	best: 0.0072444 (1000)	total: 13m 9s	remaining: 6m 33s
1200:	learn: 0.0098528	test: 0.0071043	best: 0.0071043 (1200)	total: 15m 42s	remaining: 3m 54s
1400:	learn: 0.0096632	test: 0.0069420	best: 0.0069420 (1400)	total: 18m 20s	remaining: 1m 17s
1499:	learn: 0.0095727	test: 0.0068576	best: 0.0068576 (1499)	total: 19m 38s	remaining: 0us

bestTest = 0.006857638075
bestIteration = 1499

Validation RMSE: 0.006842063436996583
Validation MSE : 4.6813832075885495e-05


In [11]:
model.save_model("catboost_model.cbm")
joblib.dump(list(X.columns), "feature_columns.pkl")

print("Saved: catboost_model.cbm + feature_columns.pkl")


Saved: catboost_model.cbm + feature_columns.pkl
